# yolo-split-computing — bottleneck validation on COCO (Colab GPU)

Trains the learned **bottleneck** and measures its **mAP cost** on full **COCO**
with the canonical **train2017 → val2017** split, on a GPU.

**Before you run:** *Runtime → Change runtime type → T4 GPU*.

Uses only **public** weights (YOLO11l COCO) and **public** data (COCO) — the
numbers are safe to publish. Use your own dataset only in a private copy.

## 1. Check the GPU

In [ ]:
import shutil, subprocess

if shutil.which('nvidia-smi'):
    print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout.strip())

try:
    import torch
    ok = torch.cuda.is_available()
except Exception:
    ok = False

if ok:
    print('GPU ready:', torch.cuda.get_device_name(0))
else:
    print('NO GPU. In Colab: Runtime > Change runtime type > Hardware accelerator > T4 GPU, then rerun.')

## 2. Install

Colab already ships a CUDA build of torch, so this is quick.

In [ ]:
!git clone --depth 1 https://github.com/dantonioluigi/yolo-split-computing.git
%cd yolo-split-computing
# Non-editable install: importable in the running Colab kernel without a restart
# (an editable `-e .` install needs a kernel restart before `import yolosplit` works).
!pip install -q .

## 3. Configuration

`TRAIN_LIMIT` caps how many train2017 images the bottleneck sees (the whole
118k is overkill and slow on the free tier). Evaluation always uses the full
val2017. Raise `EPOCHS`/`TRAIN_LIMIT` (or set `TRAIN_LIMIT = None`) for a serious
run on Colab Pro or a rented GPU.

In [ ]:
MODEL       = 'yolo11l.pt'   # public COCO-pretrained weights
IMGSZ       = 640
LATENT      = 8              # bottleneck latent channels
STRIDE      = 2              # 2 -> ~4x smaller latent (needs square val, handled below)
EPOCHS      = 30
BATCH       = 32             # lower to 16/8 if you hit CUDA OOM
LR          = 1e-3
TRAIN_LIMIT = 20000          # train2017 images for the bottleneck (None = all 118k)
DEVICE      = 'cuda'         # the GPU
OUT         = 'bottleneck.pt'

### Optional: persist to Google Drive

Colab disconnects on idle / after ~12h. Mount Drive and point `OUT` there so a
disconnect does not lose the trained bottleneck. (Uncomment to use.)

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# OUT = '/content/drive/MyDrive/yolosplit_bottleneck.pt'

## 4. Download COCO (train2017 + val2017)

First run downloads ~**20 GB** and takes ~15–20 min. Colab's free disk (~78 GB)
handles it.

In [ ]:
from pathlib import Path
from ultralytics.data.utils import check_det_dataset

data = check_det_dataset('coco.yaml')          # downloads train2017 + val2017 + labels
train_dir = str(Path(data['path']) / 'images' / 'train2017')
print('train images dir:', train_dir)
print('val defined by   :', data['val'])

## 5. Get the public detector weights

In [ ]:
import os, urllib.request
if not os.path.exists(MODEL):
    url = 'https://github.com/ultralytics/assets/releases/download/v8.3.0/' + MODEL
    urllib.request.urlretrieve(url, MODEL)
print(MODEL, f'{os.path.getsize(MODEL)/1e6:.1f} MB')

## 6. Train the bottleneck

The detector is **frozen**; only the small per-level autoencoder is trained, to
reconstruct the P3/P4/P5 backbone features through its INT8 latent (feature
distillation, with simulated INT8 noise). Watch the relative reconstruction
error — it needs to get well below ~0.2 before accuracy is retained.

In [ ]:
from ultralytics import YOLO
from yolosplit.train import train_bottleneck
from yolosplit.bottleneck import save_bottleneck

det = YOLO(MODEL).model
bottleneck, result = train_bottleneck(
    det, train_dir,
    latent_channels=LATENT, stride=STRIDE,
    epochs=EPOCHS, batch=BATCH, lr=LR, imgsz=IMGSZ,
    limit=TRAIN_LIMIT, device=DEVICE,
)
save_bottleneck(bottleneck, OUT)
print('epoch losses :', [round(x, 4) for x in result.epoch_losses])
print('recon error  :', {k: round(v, 3) for k, v in result.relative_errors.items()})
print('saved ->', OUT)

In [ ]:
import matplotlib.pyplot as plt
plt.plot(range(1, len(result.epoch_losses) + 1), result.epoch_losses, marker='o')
plt.xlabel('epoch'); plt.ylabel('normalised MSE'); plt.title('bottleneck training')
plt.grid(True); plt.show()

## 7. Measure the mAP cost on the held-out val2017

Runs the standard ultralytics validation **twice** on val2017: the unsplit model
(baseline) and the split model with the bottleneck on the wire. `rect=False`
gives the stride-2 bottleneck square inputs and makes the comparison fair.

In [ ]:
import json
from yolosplit.evaluate import compare_map
from yolosplit.bottleneck import load_bottleneck, BottleneckTransport

cmp = compare_map(
    weights=MODEL, data='coco.yaml',
    transport=BottleneckTransport(load_bottleneck(OUT), compress=True),
    imgsz=IMGSZ, device=DEVICE, rect=False,
)
d = cmp.to_dict()
print(f"baseline  mAP50={d['baseline_map50']:.3f}  mAP50-95={d['baseline_map50_95']:.3f}")
print(f"split     mAP50={d['split_map50']:.3f}  mAP50-95={d['split_map50_95']:.3f}")
print(f"delta mAP50-95 = {d['delta_map50_95']:+.3f}")
print(f"mean wire = {d['wire_mean_bytes']/1024:.1f} KB/frame")
open('eval.json', 'w').write(json.dumps(d, indent=2))

## 8. Optional: sweep for the bytes-vs-mAP Pareto curve

Trains one bottleneck per `(latent, stride)` config — heavier. Uncomment to run.

In [ ]:
# from yolosplit.sweep import run_sweep, to_markdown
# res = run_sweep(YOLO(MODEL).model, train_dir,
#                 latents=[4, 8, 16], strides=[1, 2],
#                 epochs=EPOCHS, batch=BATCH, imgsz=IMGSZ,
#                 limit=TRAIN_LIMIT, device=DEVICE)
# print(to_markdown(res))

## Notes

- **Public data only** here (COCO). Your own dataset can be used in a *private*
  copy of this notebook, but keep those numbers out of the public repo.
- **CUDA OOM?** Lower `BATCH` (16 or 8). **Session too short?** Lower
  `TRAIN_LIMIT`/`EPOCHS`, or mount Drive (cell 3) and set `OUT` there.
- **Skip the 20 GB download:** train + evaluate on a disjoint split of val2017
  only (5k images) — still a proper held-out split. Ask if you want that variant.
- A **stride-2** bottleneck needs square inputs (`rect=False`, set above); a
  **stride-1** bottleneck has no such constraint but ships ~4x larger latents.